# 1 环量理论与解析性

本 Notebook 对应项目二任务 1.1-1.3，目标是：

1. 给出复势函数与复速度的基本定义。
2. 用数值方式验证 C-R 条件在圆柱外区域近似成立。
3. 验证环量积分结果与理论值的一致性。

核心模型：

$$
\Phi(z)=U\left(z+\frac{a^2}{z}\right)+\frac{i\Gamma}{2\pi}\ln z
$$

In [5]:
import sys
from pathlib import Path

import numpy as np

root = Path.cwd()
if not (root / 'src').exists():
    for p in [root, *root.parents]:
        if (p / 'src').exists() and (p / 'requirements.txt').exists():
            root = p
            break
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.ui.simulator import FlowSimulator
from src.utils.cr_verify import verify_cr

U = 1.0
a = 1.0
gamma = 3.0

sim = FlowSimulator(U=U, a=a, gamma=gamma, n=201)
out = sim.run()

dx = (sim.x_max - sim.x_min) / (sim.n - 1)
dy = (sim.y_max - sim.y_min) / (sim.n - 1)
cr_stats = verify_cr(out['phi'], out['psi'], dx=dx, dy=dy, mask=out['valid_mask'])
cr_stats

{'max_abs_r1': 0.0019398692922130145,
 'max_abs_r2': 50.000806424836426,
 'mean_abs_r1': 0.00010711067836815082,
 'mean_abs_r2': 0.1799249044223153}

## 环量积分验证

理论上在圆周 $|z|=a$ 上有：

$$
\Gamma_{calc}=\mathrm{Re}\oint \frac{d\Phi}{dz}dz = -\Gamma
$$

下面使用离散积分进行数值验证。

In [4]:
theta = np.linspace(0, 2*np.pi, 4000, endpoint=False)
z = a * np.exp(1j * theta)
dz = 1j * a * np.exp(1j * theta) * (theta[1] - theta[0])

dwdz = U * (1 - a**2 / z**2) + 1j * gamma / (2*np.pi*z)
gamma_calc = np.real(np.sum(dwdz * dz))

print({'gamma_input': gamma, 'gamma_calc': gamma_calc, 'target': -gamma})

{'gamma_input': 3.0, 'gamma_calc': np.float64(-3.0), 'target': -3.0}
